# End-to-End Python API Workflow

This notebook runs the same stack directly through the package APIs. It also compares the same congestion-focused scenario under `idealized` and `reserved_edges` execution.

In [1]:
from __future__ import annotations

import csv
import json
import os
import sys
from dataclasses import replace
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src").exists(), "Run this notebook from the repository root or notebooks/ directory."
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)

from warehouse_sim.config import load_experiment_config
from warehouse_sim.simulation import run_benchmark_from_path
from warehouse_sim.simulation.runner import run_experiment_from_config

OUTPUT_ROOT = REPO_ROOT / "outputs" / "notebooks" / "e2e_python_api"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(REPO_ROOT)
print(OUTPUT_ROOT)


${REPO_ROOT}
${REPO_ROOT}/outputs/notebooks/e2e_python_api


In [2]:
config_path = REPO_ROOT / "configs" / "scenarios" / "narrow_bottleneck.toml"
base_config = load_experiment_config(config_path)

idealized_config = replace(
    base_config,
    name=f"{base_config.name}_idealized",
    simulation=replace(base_config.simulation, execution_model="idealized"),
)
reserved_edges_config = replace(
    base_config,
    name=f"{base_config.name}_reserved_edges",
    simulation=replace(base_config.simulation, execution_model="reserved_edges"),
)

idealized_result, idealized_written = run_experiment_from_config(
    idealized_config,
    output_dir_override=OUTPUT_ROOT / "narrow_bottleneck_idealized",
)
reserved_result, reserved_written = run_experiment_from_config(
    reserved_edges_config,
    output_dir_override=OUTPUT_ROOT / "narrow_bottleneck_reserved_edges",
)

comparison = {
    "idealized": {
        "tasks_completed": idealized_result.metrics.tasks_completed,
        "makespan": idealized_result.metrics.makespan,
        "congestion_delay_total": idealized_result.metrics.congestion_delay_total,
        "blocked_traversal_events_total": idealized_result.metrics.blocked_traversal_events_total,
    },
    "reserved_edges": {
        "tasks_completed": reserved_result.metrics.tasks_completed,
        "makespan": reserved_result.metrics.makespan,
        "congestion_delay_total": reserved_result.metrics.congestion_delay_total,
        "blocked_traversal_events_total": reserved_result.metrics.blocked_traversal_events_total,
    },
}
comparison


{'idealized': {'tasks_completed': 11,
  'makespan': 637.8644397795285,
  'congestion_delay_total': 0.0,
  'blocked_traversal_events_total': 0},
 'reserved_edges': {'tasks_completed': 11,
  'makespan': 637.8644397795285,
  'congestion_delay_total': 0.0,
  'blocked_traversal_events_total': 0}}

In [3]:
with reserved_written["executions"].open() as handle:
    execution_rows = list(csv.DictReader(handle))

execution_rows[:2]


[{'task_id': 'task_1',
  'robot_id': 'robot_1',
  'release_time': '10.330585757814186',
  'assigned_at': '10.330585757814186',
  'pickup_arrival_time': '14.330585757814186',
  'service_start_time': '14.330585757814186',
  'completion_time': '42.330585757814184',
  'waiting_time': '0.0',
  'turnaround_time': '32.0',
  'execution_model': 'reserved_edges',
  'travel_to_pickup_time': '4.0',
  'travel_to_pickup_distance': '4.0',
  'travel_to_pickup_ideal_time': '4.0',
  'travel_to_pickup_wait_time': '0.0',
  'travel_to_pickup_blocked_events': '0',
  'travel_to_pickup_path_nodes': "('r4_c0', 'r3_c0', 'r2_c0', 'r1_c0', 'r0_c0')",
  'travel_to_pickup_path_arcs': "('r4_c0->r3_c0', 'r3_c0->r2_c0', 'r2_c0->r1_c0', 'r1_c0->r0_c0')",
  'travel_to_dropoff_time': '8.0',
  'travel_to_dropoff_distance': '8.0',
  'travel_to_dropoff_ideal_time': '8.0',
  'travel_to_dropoff_wait_time': '0.0',
  'travel_to_dropoff_blocked_events': '0',
  'travel_to_dropoff_path_nodes': "('r0_c0', 'r0_c1', 'r0_c2', 'r0_c3',

In [4]:
benchmark_written = run_benchmark_from_path(
    REPO_ROOT / "configs" / "congestion_policy_benchmark.toml",
    benchmark_root_override=OUTPUT_ROOT / "congestion_benchmark",
    force_write_plots=False,
)

benchmark_payload = json.loads(benchmark_written["summary_json"].read_text())
print("benchmark name", benchmark_payload["benchmark_name"])
print("runs", len(benchmark_payload["runs"]))
benchmark_payload["best_by_scenario"]


benchmark name congestion_policy_benchmark
runs 12


{'narrow_bottleneck': {'scenario_name': 'narrow_bottleneck',
  'scenario_config': '${REPO_ROOT}/configs/scenarios/narrow_bottleneck.toml',
  'seed': 23,
  'policy': 'fifo',
  'execution_model': 'reserved_edges',
  'tasks_generated': 11,
  'tasks_completed': 11,
  'tasks_unassigned': 0,
  'average_waiting_time': 0.0,
  'average_turnaround_time': 34.90909090909091,
  'average_travel_distance_per_task': 14.909090909090908,
  'realized_travel_time_total': 164.0,
  'realized_travel_distance_total': 164.0,
  'congestion_delay_total': 0.0,
  'average_congestion_delay_per_completed_task': 0.0,
  'blocked_traversal_events_total': 0,
  'average_queue_length': 0.0,
  'throughput_per_hour': 62.082156537347245,
  'makespan': 637.8644397795285,
  'summary_path': '${REPO_ROOT}/outputs/notebooks/e2e_python_api/congestion_benchmark/narrow_bottleneck/seed_23/fifo/summary.json'},
 'high_fleet_density': {'scenario_name': 'high_fleet_density',
  'scenario_config': '${REPO_ROOT}/configs/scenarios/high_fleet

In [5]:
summary_paths = {
    "idealized_summary": str(idealized_written["summary"]),
    "reserved_edges_summary": str(reserved_written["summary"]),
    "benchmark_summary": str(benchmark_written["summary_json"]),
}
summary_paths


{'idealized_summary': '${REPO_ROOT}/outputs/notebooks/e2e_python_api/narrow_bottleneck_idealized/summary.json',
 'reserved_edges_summary': '${REPO_ROOT}/outputs/notebooks/e2e_python_api/narrow_bottleneck_reserved_edges/summary.json',
 'benchmark_summary': '${REPO_ROOT}/outputs/notebooks/e2e_python_api/congestion_benchmark/benchmark_summary.json'}